# Іспит (90 хв): Міні‑проєкт у Jupyter Notebook

- Курс: "Основи генеративного ШІ" (Комп'ютерна інженерія, 2 курс)
- Використовуйте ті самі інструменти/моделі, що в практичних (GitHub Models/Azure AI Inference).
- Назва файлу перед здачею: `Прізвище І.Б._varNN.ipynb` (змініть у назві цього файлу перед завантаженням).
- Після завершення — завантажити на Google‑диск, вказаний викладачем.

## Структура іспиту
- **[0]** Титул і варіант (ПІБ, група, варіант, модель, endpoint)
- **[1]** Підготовка середовища (5-10 хв)
- **[2]** Базова генерація (15-20 хв)
- **[2b]** Діалоговий режим чат-бота (10-15 хв)
- **[3]** Покращення промпту (20-25 хв)
- **[4]** Індивідуальна частина (25-30 хв)
- **[5]** Рефлексія (5-10 хв)

## Індивідуальні варіанти
Детальний список варіантів та інструкції див. у файлі `Іспит_мініпроєкт_GenAI_осінь_2025.md`

**Блоки варіантів:**
- **Блок A (1-10)**: Основи комп'ютерних систем
- **Блок B (11-20)**: Програмне забезпечення та мережі  
- **Блок C (21-30)**: Інтернет речей та embedded системи

**Стандартна структура кожного варіанту:** генеруйте 3+ приклади → валідуйте формат → порівняйте результати → зробіть висновки.

Вкажіть нижче свій ПІБ, групу, варіант, модель та endpoint.

## [0] Титул і варіант
- ПІБ: Яценко Матвій Сергійович
- Група: КІ-25
- Варіант: 17
- Модель: gpt-4o-mini
- Endpoint: https://models.inference.ai.azure.com
- Час початку: 15:05

## Важливо
- Перевірте свій варіант у файлі `Іспит_мініпроєкт_GenAI_осінь_2025.md`
- Стандартна структура кожного варіанту: генеруйте 3+ приклади → валідуйте формат → порівняйте результати → зробіть висновки
- Усі варіанти мають однаковий рівень складності та однакову структуру виконання

In [8]:
%pip install -q pandas jsonschema python-dotenv azure-ai-inference


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
# [1] Підготовка середовища: імпорти, .env, перевірка токена та клієнта
import os, time, json
import pandas 
from typing import Dict, Any, Tuple

# .env (як у практичних)
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
ENDPOINT = os.getenv('AZUREAI_INFERENCE_ENDPOINT', 'https://models.inference.ai.azure.com')
MODEL = os.getenv('GENAI_MODEL', 'gpt-4o-mini')
assert GITHUB_TOKEN, 'GITHUB_TOKEN не знайдено. Додайте у .env'
print('✅ Токен завантажено:', bool(GITHUB_TOKEN))
print('➡️ Endpoint:', ENDPOINT)
print('➡️ Model:', MODEL)

# Підготовка клієнта: azure-ai-inference або fallback на requests (узгоджено з практичними)
client_mode = 'requests'
try:
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    _client = ChatCompletionsClient(endpoint=ENDPOINT, credential=AzureKeyCredential(GITHUB_TOKEN))
    client_mode = 'azure-ai-inference'
except Exception:
    import requests
    _client = None

print('➡️ Client mode:', client_mode)

def ask_llm(system: str, user: str, temperature: float = 0.7, max_tokens: int = 256) -> Tuple[str, Dict[str, Any], float]:
    start = time.time()
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    if client_mode == 'azure-ai-inference':
        # Виклик через SDK (див. ПР3/ПР4/ПР6/ПР7)
        resp = _client.complete(messages=messages, model=MODEL, temperature=temperature, max_tokens=max_tokens)
        text = resp.choices[0].message.content if hasattr(resp.choices[0].message, 'content') else resp.choices[0].message['content']
        usage = getattr(resp, 'usage', {}) or {}
    else:
        import requests
        url = ENDPOINT.rstrip('/') + '/chat/completions'
        headers = {
            'Authorization': f'Bearer {GITHUB_TOKEN}',
            'Content-Type': 'application/json'
        }
        payload = {
            'model': MODEL,
            'messages': messages,
            'temperature': temperature,
            'max_tokens': max_tokens
        }
        r = requests.post(url, headers=headers, json=payload, timeout=60)
        r.raise_for_status()
        data = r.json()
        text = data['choices'][0]['message']['content']
        usage = data.get('usage', {})
    latency = time.time() - start
    return text, usage, latency

# Швидкий sanity-check (1 короткий запит)
txt, usg, lat = ask_llm(
    system="Ви лаконічний помічник.",
    user="Скажи 'готово'.",
    temperature=0.0, max_tokens=16
)
print('LLM OK, latency:', round(lat, 2), 's')
print('Response:', txt)


✅ Токен завантажено: True
➡️ Endpoint: https://models.inference.ai.azure.com
➡️ Model: gpt-4o-mini
➡️ Client mode: azure-ai-inference
LLM OK, latency: 1.6 s
Response: Готово.


## [2] Базова генерація (15-20 хв)
Створіть базовий промпт для вашої теми (згідно з варіантом). Прогоніть серію запусків із різними `temperature`/`max_tokens` (мін. 3). Зафіксуйте latency/usage та коротко оцініть якість.

**Приклад для варіанту 1 (Алгоритми сортування):**
- Тема: порівняння алгоритмів сортування
- Базовий промпт: "Опиши 3 алгоритми сортування: bubble sort, quick sort, merge sort"
- Параметри: temperature=[0.2, 0.7, 0.9], max_tokens=[256, 256, 512]

**Вимоги:**
- Використовуйте тему вашого варіанту
- Зробіть мінімум 3 запити з різними параметрами
- Запишіть результати у `results_base`
- Оцініть якість відповідей
- Зробіть висновки про вплив параметрів на результат

In [11]:
# Базова генерація для варіанту 17 (Відеоконференції)
system = 'Ви експерт з програмного забезпечення та онлайн-комунікацій. Пояснюйте чітко і структуровано.'
user_base = '''Опиши 3 платформи для відеоконференцій: Zoom, Teams, Google Meet. 
Для кожної вкажи: platform (назва), max_participants (максимальна кількість учасників), quality (якість зв'язку), price (ціна). 
Додай порівняння цих платформ для цілей навчання.'''

settings = [(0.2, 256), (0.7, 256), (0.9, 512)]
results_base = []

for t, mx in settings:
    text, usage, latency = ask_llm(system, user_base, temperature=t, max_tokens=mx)
    results_base.append({
        'temperature': t, 
        'max_tokens': mx, 
        'latency_s': latency, 
        'usage': usage, 
        'text': text[:400]
    })
    print(f"T={t}, Tokens={mx}: {text[:100]}...\n")

print("\n=== Результати базової генерації ===")
for i, res in enumerate(results_base):
    print(f"\n{i+1}. Temp={res['temperature']}, Latency={res['latency_s']:.2f}s")
    print(f"   Usage: {res['usage']}")
    print(f"   Text: {res['text'][:150]}...")

print("\n=== Висновки про вплив параметрів ===")
print("- Temperature 0.2: найбільш точні технічні дані, стабільна структура.")
print("- Temperature 0.7: хороший баланс між сухими фактами та розгорнутим висновком для навчання.")
print("- Temperature 0.9: більш креативні описи, але може страждати чіткість формату.")
print("- Latency традиційно зростає при генерації більшої кількості токенів.")

T=0.2, Tokens=256: Ось опис трьох популярних платформ для відеоконференцій: Zoom, Microsoft Teams та Google Meet.

### ...

T=0.7, Tokens=256: Ось опис трьох популярних платформ для відеоконференцій: Zoom, Microsoft Teams, Google Meet, разом і...

T=0.9, Tokens=512: ### 1. Zoom
- **Platform**: Zoom
- **Max Participants**: 100 (безкоштовний план), до 1000 (платні пл...


=== Результати базової генерації ===

1. Temp=0.2, Latency=4.41s
   Usage: {'completion_tokens': 256, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens': 118, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'total_tokens': 374}
   Text: Ось опис трьох популярних платформ для відеоконференцій: Zoom, Microsoft Teams та Google Meet.

### 1. Zoom
- **platform**: Zoom
- **max_participants*...

2. Temp=0.7, Latency=3.85s
   Usage: {'completion_tokens': 256, 'completion_tokens_details': {'accepted_predictio

## [2b] Діалоговий режим чат-бота (5–10 хв)
Реалізуйте простий чат з історією повідомлень (`messages`). Використовуйте той самий клієнт/endpoint і функцію `ask_llm`.

Пояснення для студентів:
- Чат не запускається автоматично, щоб ноутбук не «зависав» у середовищах без stdin.
- Для запуску встановіть `RUN_CHAT = True` у кодовій клітинці нижче.
- Команди: `/exit` — вихід; `/reset` — очистити історію.
- Обмеження `MAX_TURNS` запобігає нескінченним діалогам.
- Рекомендовані параметри: `temperature≈0.3`, `max_tokens≈300` (можна змінювати).

In [6]:
# Діалоговий режим: безпечний чат-цикл з обмеженням ходів
import sys

messages = [{"role": "system", "content": "Ви помічник у діалоговому режимі. Відповідайте коротко і по суті."}]

def chat_once(user_text: str, temperature: float = 0.4, max_tokens: int = 600):
    global messages
    messages.append({"role": "user", "content": user_text})
    sys_prompt = next((m["content"] for m in reversed(messages) if m["role"] == "system"), "")
    text, usage, latency = ask_llm(sys_prompt, user_text, temperature=temperature, max_tokens=max_tokens)
    messages.append({"role": "assistant", "content": text})
    print(text)
    sys.stdout.flush()

RUN_CHAT = True  # встановіть True, щоб запустити чат
MAX_TURNS = 20    # безпечна межа кількості ходів

if RUN_CHAT:
    print("Чат-режим. Команди: /exit — вихід, /reset — очистити історію")
    turns = 0
    while turns < MAX_TURNS:
        try:
            user_in = input("you> ").strip()
        except EOFError:
            break
        if not user_in:
            continue
        if user_in == "/exit":
            print("Вихід з чат-режиму.")
            break
        if user_in == "/reset":
            messages = [{"role": "system", "content": "Ви помічник у діалоговому режимі. Відповідайте коротко і по суті."}]
            print("Історію очищено.")
            continue
        chat_once(user_in)
        turns += 1
    if turns >= MAX_TURNS:
        print("Досягнуто межі ходів (MAX_TURNS). Чат зупинено для безпеки.")

Чат-режим. Команди: /exit — вихід, /reset — очистити історію
Привіт! Як можу допомогти?
Фондовий ринок — це платформа, де купуються і продаються акції, облігації та інші фінансові інструменти. Він забезпечує компаніям можливість залучати капітал, а інвесторам — можливість отримувати прибуток від своїх інвестицій.

**Основні аспекти фондового ринку:**

1. **Учасники:**
   - **Емітенти:** Компанії, які випускають акції чи облігації.
   - **Інвестори:** Особи або організації, які купують цінні папери.
   - **Брокери:** Посередники, які здійснюють торги між покупцями і продавцями.

2. **Типи ринків:**
   - **Первинний ринок:** Місце, де компанії вперше пропонують свої акції (IPO).
   - **Вторинний ринок:** Платформа для торгівлі вже випущеними цінними паперами.

3. **Функціонування:**
   - **Ціноутворення:** Ціни акцій формуються на основі попиту та пропозиції.
   - **Торгові платформи:** Торги відбуваються на біржах (наприклад, NYSE, NASDAQ) або через позабіржові системи.

4. **Регулюванн


## [3] Покращення промпту (20-25 хв)
Додайте системний контекст, few-shot приклади, формат-контракт (JSON) та правила безпеки. Повторіть 1-2 запуски і порівняйте з [2].

**Приклад покращень для варіанту 1:**
- Системний контекст: детальна інструкція про JSON формат
- Few-shot: 1 приклад правильної структури
- Формат-контракт: сувора JSON схема
- Правила безпеки: перевірка синтаксису коду

**Вимоги:**
- Додайте мінімум 2 прийоми покращення промпту
- Вкажіть очікуваний формат відповіді (JSON)
- Додайте перевірку валідності результату
- Порівняйте якість з базовою генерацією
- Проведіть 2-3 запуски для стабільності результатів
- Проаналізуйте вплив кожного покращення

In [12]:
# Покращена генерація для варіанту 17

system_improved = (
    'Ви експерт з онлайн-комунікацій. Генеруйте відповіді виключно у форматі JSON. '
    'Жодної Markdown-розмітки (без ```json), лише чистий JSON. '
    'Структура має містити масив "platforms" та текстове поле "education_comparison".'
)

# Few-shot приклад
fewshot = '''Приклад правильної відповіді:
{
  "platforms": [
    {
      "platform": "Skype",
      "max_participants": "100",
      "quality": "Середня",
      "price": "Безкоштовно"
    }
  ],
  "education_comparison": "Skype підходить для невеликих репетиторських занять, але поступається спеціалізованим платформам для великих лекцій."
}

Твоє завдання:
'''

user_improved = '''Опиши 3 платформи: Zoom, Teams, Google Meet. 
Відповідь у форматі JSON з масивом "platforms" (поля: platform, max_participants, quality, price) 
та полем "education_comparison", де порівнюються ці платформи для цілей навчання.'''

# Проведення 3 запусків для стабільності
improved_results = []
for attempt in range(3):
    print(f"\n=== Покращена генерація, спроба {attempt + 1} ===")
    text_i, usage_i, lat_i = ask_llm(system_improved, fewshot + user_improved, temperature=0.3, max_tokens=600)
    
    # Очищення від можливих Markdown-блоків (перестраховка)
    if text_i.startswith('```json'):
        text_i = text_i[7:-3].strip()
    elif text_i.startswith('```'):
        text_i = text_i[3:-3].strip()
        
    print('Покращена відповідь:')
    print(text_i[:300] + "..." if len(text_i) > 300 else text_i)
    print(f'Latency: {lat_i:.2f}s')
    
    try:
        import json
        data = json.loads(text_i)
        print('✅ JSON успішно розпарсено')
        
        platforms = data.get('platforms', [])
        valid_count = sum(1 for p in platforms if all(k in p for k in ['platform', 'max_participants', 'quality', 'price']))
        print(f'Платформ з правильною структурою: {valid_count}/{len(platforms)}')
        
        improved_results.append({
            'attempt': attempt + 1, 'success': True, 'data': data,
            'usage': usage_i, 'latency': lat_i
        })
    except Exception as e:
        print(f'❌ Помилка парсингу JSON: {e}')


=== Покращена генерація, спроба 1 ===


Покращена відповідь:
{
  "platforms": [
    {
      "platform": "Zoom",
      "max_participants": "1000",
      "quality": "Висока",
      "price": "Безкоштовно з обмеженнями, платні плани доступні"
    },
    {
      "platform": "Teams",
      "max_participants": "300",
      "quality": "Висока",
      "price": "Безкош...
Latency: 3.27s
✅ JSON успішно розпарсено
Платформ з правильною структурою: 3/3

=== Покращена генерація, спроба 2 ===
Покращена відповідь:
{
  "platforms": [
    {
      "platform": "Zoom",
      "max_participants": "1000",
      "quality": "Висока",
      "price": "Безкоштовно (з обмеженнями) / Платні плани"
    },
    {
      "platform": "Teams",
      "max_participants": "1000",
      "quality": "Висока",
      "price": "Безкоштовно...
Latency: 3.08s
✅ JSON успішно розпарсено
Платформ з правильною структурою: 3/3

=== Покращена генерація, спроба 3 ===
Покращена відповідь:
{
  "platforms": [
    {
      "platform": "Zoom",
      "max_participants": "1000",
      "q

## [4] Індивідуальна частина (25-30 хв)

### Структура виконання:
1. **Генерація**: 3+ приклади з `temperature=0.4`, `max_tokens=600`
2. **Валідація**: JSON схема + `jsonschema`
3. **Аналіз**: pandas DataFrame + порівняння
4. **Висновки**: стабільність, якість, проблеми

### Індивідуальна частина для вашого варіанту (1-30)
### ЗАМІНІТЬ КОД НИЖЧЕ ВІДПОВІДНО ДО ВАШОГО ВАРІАНТУ

### Варінт 17 Відеоконференції
import jsonschema, pandas as pd

### Крок 1: Промпти
system_variant = 'Ви експерт з комп\'ютерного обладнання. Генеруйте у JSON форматі.'
user_variant = 'Опишіть 3 платформи (Zoom, Teams, Google Meet) у JSON {platform, max_participants, quality, price}. Порівняйте для навчання.'
fewshot_variant = '''Приклад:
{
  "components": [
    {"name": "Intel i7", "function": "Обробка даних", "speed": "3.5 GHz", "cost": "200-300 USD"}
  ]
}'''

### Крок 2: JSON схема
variant_schema = {
    "type": "object",
    "properties": {
        "components": {
            "type": "array", "minItems": 3,
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "function": {"type": "string"},
                    "speed": {"type": "string"},
                    "cost": {"type": "string"}
                },
                "required": ["name", "function", "speed", "cost"]
            }
        }
    },
    "required": ["components"]
}

### Крок 3: Генерація та валідація
variant_results = []
successful_attempts = 0

for attempt in range(3):
    print(f"\nСпроба {attempt + 1}")
    
    # Генерація
    text_v, usage_v, lat_v = ask_llm(
        system_variant, fewshot_variant + user_variant, 
        temperature=0.4, max_tokens=600
    )
    
    print(f'Latency: {lat_v:.2f}s')
    
    # Валідація
    try:
        data = json.loads(text_v)
        jsonschema.validate(data, variant_schema)
        
        components = data.get("components", [])
        valid_objects = sum(1 for comp in components 
                          if all(field in comp and comp[field] for field in ["name", "function", "speed", "cost"]))
        
        print(f'✅ JSON валідний: {valid_objects}/{len(components)} об\'єктів')
        
        successful_attempts += 1
        variant_results.append({
            'attempt': attempt + 1, 'success': True, 'data': data,
            'valid_objects': valid_objects, 'total_objects': len(components),
            'usage': usage_v, 'latency': lat_v
        })
        
    except Exception as e:
        print(f'❌ Помилка: {e}')
        variant_results.append({
            'attempt': attempt + 1, 'success': False, 'error': str(e),
            'usage': usage_v, 'latency': lat_v
        })

### Крок 4: Аналіз результатів
print("\n" + "="*50)
print("ПОРІВНЯЛЬНИЙ АНАЛІЗ")
print("="*50)

if successful_attempts > 0:
    all_components = []
    for result in variant_results:
        if result['success']:
            all_components.extend(result['data']['components'])
    
    df = pd.DataFrame(all_components)
    print("📊 Таблиця порівняння:")
    print(df[['name', 'speed', 'cost']].to_string(index=False))
    
    # Статистика
    print(f"\n📈 Статистика:")
    print(f"- Успішних спроб: {successful_attempts}/3 ({successful_attempts/3*100:.1f}%)")
    print(f"- Компонентів: {len(all_components)}")
    print(f"- Середній latency: {sum(r['latency'] for r in variant_results)/len(variant_results):.2f}s")

# Крок 5: Висновки
print(f"\n💡 ВИСНОВКИ:")
print(f"- Стабільність: {successful_attempts}/3 спроб успішних")
print(f"- JSON-контракт працює ефективно")
print(f"- Temperature 0.4 оптимальний")

if successful_attempts == 3:
    print("✅ Генерація стабільна")
elif successful_attempts >= 2:
    print("⚠️ Генерація частково стабільна")
else:
    print("❌ Генерація нестабільна")

### Фінальні результати
final_results = {
    "variant": 1,  # ЗАМІНІТЬ НА ВАШ ВАРІАНТ
    "attempts": 3, "successful": successful_attempts,
    "success_rate": successful_attempts / 3 * 100,
    "data": all_components if successful_attempts > 0 else []
}

print(f"\n📋 Результати:")
print(f"Варіант: {final_results['variant']}")
print(f"Успішність: {final_results['success_rate']:.1f}%")

### ЗАМІНІТЬ НА ВАШ ВАРІАНТ:
### 1. system_variant, user_variant, fewshot_variant
### 2. variant_schema для вашої структури  
### 3. Адапуйте аналіз під вашу тему
### 4. Замініть номер варіанту у final_results

In [16]:
import jsonschema, pandas as pd
import json

### Крок 1: Промпти
system_variant = (
    'Ви експерт з дистанційного навчання. Генеруйте відповіді виключно у форматі чистого JSON. '
    'Починайте з { і закінчуйте }.'
)
fewshot_variant = '''Приклад:
{
  "platforms": [
    {"platform": "Discord", "max_participants": "25 (відео)", "quality": "Висока", "price": "Безкоштовно"}
  ],
  "education_comparison": "Discord популярний серед студентів для неформальних груп, але рідко використовується як офіційна платформа."
}

Твоє завдання:
'''
user_variant = '''Опиши 3 платформи: Zoom, Teams, Google Meet. 
Формат JSON: масив "platforms" (поля: platform, max_participants, quality, price) та поле "education_comparison".'''

### Крок 2: JSON схема
variant_schema = {
    "type": "object",
    "properties": {
        "platforms": {
            "type": "array", "minItems": 3,
            "items": {
                "type": "object",
                "properties": {
                    "platform": {"type": "string"},
                    "max_participants": {"type": "string"},
                    "quality": {"type": "string"},
                    "price": {"type": "string"}
                },
                "required": ["platform", "max_participants", "quality", "price"]
            }
        },
        "education_comparison": {"type": "string"}
    },
    "required": ["platforms", "education_comparison"]
}

### Крок 3: Генерація та валідація
variant_results = []
successful_attempts = 0

for attempt in range(3):
    print(f"\nСпроба {attempt + 1}")
    text_v, usage_v, lat_v = ask_llm(system_variant, fewshot_variant + user_variant, temperature=0.2, max_tokens=600)
    
    # НАДІЙНЕ ОЧИЩЕННЯ JSON ВІД МАРКДАУНУ
    text_clean = text_v.strip()
    start_idx = text_clean.find('{')
    end_idx = text_clean.rfind('}')
    
    if start_idx != -1 and end_idx != -1:
        text_clean = text_clean[start_idx:end_idx+1]
        
    try:
        data = json.loads(text_clean)
        jsonschema.validate(data, variant_schema)
        
        platforms = data.get("platforms", [])
        valid_objects = len(platforms)
        print(f"✅ JSON валідний: {valid_objects}/{len(platforms)} об'єктів")
        
        successful_attempts += 1
        variant_results.append({
            'success': True, 'data': data,
            'latency': lat_v
        })
        
    except Exception as e:
        print(f"❌ Помилка: {e}")
        # Виводимо початок тексту, щоб зрозуміти, що саме видала модель
        print(f"Отриманий текст: {text_v[:150]}...")

### Крок 4: Аналіз результатів
print("\n" + "="*50)
print("ПОРІВНЯЛЬНИЙ АНАЛІЗ ПЛАТФОРМ")
print("="*50)

if successful_attempts > 0:
    # Беремо дані з першої успішної спроби для таблиці
    best_data = variant_results[0]['data']
    df = pd.DataFrame(best_data['platforms'])
    print("📊 Таблиця порівняння:")
    print(df[['platform', 'max_participants', 'quality', 'price']].to_string(index=False))
    
    print(f"\n📝 Висновок для навчання:\n{best_data['education_comparison']}")
    print(f"\n📈 Статистика:\n- Успішних спроб: {successful_attempts}/3")
else:
    print("❌ Жодної успішної спроби. Неможливо побудувати таблицю.")

### Фінальні результати
final_results = {
    "variant": 17,
    "topic": "Відеоконференції",
    "attempts": 3, 
    "successful": successful_attempts,
    "success_rate": successful_attempts / 3 * 100
}

print(f"\n📋 Результати: Варіант {final_results['variant']} | Успішність: {final_results['success_rate']:.1f}%")


Спроба 1


✅ JSON валідний: 3/3 об'єктів

Спроба 2
✅ JSON валідний: 3/3 об'єктів

Спроба 3
✅ JSON валідний: 3/3 об'єктів

ПОРІВНЯЛЬНИЙ АНАЛІЗ ПЛАТФОРМ
📊 Таблиця порівняння:
   platform                    max_participants quality                                            price
       Zoom 100 (безкоштовно), до 1000 (платно)  Висока Безкоштовно з обмеженнями, платні плани доступні
      Teams 300 (безкоштовно), до 1000 (платно)  Висока Безкоштовно з обмеженнями, платні плани доступні
Google Meet  100 (безкоштовно), до 500 (платно)  Висока Безкоштовно з обмеженнями, платні плани доступні

📝 Висновок для навчання:
Zoom та Teams широко використовуються в освітніх установах для онлайн-уроків, тоді як Google Meet популярний серед користувачів Google Workspace.

📈 Статистика:
- Успішних спроб: 3/3

📋 Результати: Варіант 17 | Успішність: 100.0%


## [5] Рефлексія (5-10 хв)
Коротко опишіть виконання завдання:

### Що спрацювало:
- Які промпт-інженерінгові техніки були найефективнішими?
- Які параметри (temperature, max_tokens) дали найкращі результати?
- Чи вдалося отримати структуровану відповідь у потрібному форматі?
- Які компоненти системи (клієнт, валідація, порівняння) працювали добре?

### Що не спрацювало:
- Які труднощі виникали з генерацією JSON?
- Чи доводилося робити кілька спроб для отримання коректного формату?
- Які обмеження моделі ви помітили?
- Які технічні проблеми виникали (API, парсинг, валідація)?

### Найбільший ефект дали:
- Детальний системний промпт з чіткими інструкціями
- Few-shot приклади для демонстрації формату
- Temperature = 0.3 для стабільності результатів
- Валідація JSON для перевірки коректності
- Більший час на виконання дозволив зробити більше спроб

### Обмеження та висновки:
- Складність отримання синтаксично коректного коду
- Необхідність кількох спроб для складних структур
- Важливість точного формулювання вимог до формату
- Роль валідації в забезпеченні якості результатів
- Вплив часу на якість та стабільність результатів

### Загальна оцінка:
- Чи вдалося виконати завдання повністю?
- Які б ви зробили покращення при повторному виконанні?
- Що ви навчилися про промпт-інженеринг для технічних завдань?
- Чи достатньо було 90 хвилин для якісного виконання?

# Завершення роботи

## 📋 Інструкції для здачі
1. **Перейменуйте файл**: `Прізвище І.Б._varNN.ipynb`
2. **Перевірте всі секції**: [0] → [1] → [2] → [2b] → [3] → [4] → [5]
3. **Запустіть всі клітинки**: Перевірте, що все працює коректно
4. **Очистіть вивід**: За бажанням очистьте довгі виводи для економії місця
5. **Збережіть файл**: Переконайтеся, що всі зміни збережено

## ✅ Checklist перед здачею
- [ ] ПІБ, група, варіант заповнені в секції [0]
- [ ] GITHUB_TOKEN працює (секція [1])
- [ ] Базова генерація виконана з 3+ параметрами (секція [2])
- [ ] Діалоговий режим готовий (секція [2b])
- [ ] Покращення промпту з few-shot та JSON (секція [3])
- [ ] Індивідуальна частина відповідно до варіанту (секція [4])
- [ ] Рефлексія з аналізом результатів (секція [5])
- [ ] Файл перейменовано правильно

## 🚀 Завантаження
- Завантажте файл на Google-диск, вказаний викладачем
- Будьте готові до короткої усної демонстрації (3-5 хв)

---
**Успіхів на іспиті!** 🎯